# 06 · Results and benchmarks
Experiments run every controller on 12 scenarios of 2026 with 3 random seeds each
(`python scripts/run_experiments.py`). Baselines: SUMO's **fixed** plan and **actuated** control.
TraffiX variants: junction agents only, rules only (no models), and the full system with each
traffic light agent.

In [ ]:
import os, sys, warnings
sys.path.insert(0, os.path.abspath(".."))          # the project root, so `traffix` can be imported
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Image, display
pd.set_option("display.max_colwidth", 160); pd.set_option("display.width", 200)
from traffix.core import config
from traffix.evaluation import notebook_tools as nt

In [ ]:
res = nt.load_results()
if res is None:
    print("No results yet: run  python scripts/run_experiments.py")
else:
    order = [c for c in ["fixed", "actuated", "junction_only", "traffix_rules", "traffix", "traffix_predictive", "traffix_rl"] if c in set(res.controller)]
    print(f"{len(res)} simulations: {res.scenario.nunique()} scenarios × {res.controller.nunique()} controllers × {res.seed.nunique()} seeds")

## 1. Car and pedestrian waiting (mean over seeds)

In [ ]:
if res is not None:
    for col, title in [("veh_mean_wait_s", "Car wait (s)"), ("ped_mean_wait_s", "Pedestrian wait (s)")]:
        print(title); display(res.pivot_table(index="scenario", columns="controller", values=col)[order].round(1))

## 2. Safety: conflicting greens while someone is still crossing (truncations per run)

In [ ]:
if res is not None:
    display(res.pivot_table(index="scenario", columns="controller", values="clearance_truncations")[order].round(1))

## 3. TraffiX against SUMO's actuated control, with paired t-tests over seeds
A difference is called significant when p < 0.05.

In [ ]:
from scipy import stats
if res is not None:
    rows = []
    for sc, g in res.groupby("scenario"):
        a = g[g.controller == "traffix"].set_index("seed"); b = g[g.controller == "actuated"].set_index("seed")
        seeds = a.index.intersection(b.index)
        for col, label in [("veh_mean_wait_s", "car wait"), ("ped_mean_wait_s", "pedestrian wait")]:
            x, z = a.loc[seeds, col], b.loc[seeds, col]
            p = stats.ttest_rel(x, z).pvalue if len(seeds) > 1 else np.nan
            rows.append({"scenario": sc, "measure": label, "actuated": z.mean(), "traffix": x.mean(),
                         "change %": (x.mean() - z.mean()) / z.mean() * 100, "p": p, "significant": p < 0.05})
    display(pd.DataFrame(rows).round(3).pivot_table(index="scenario", columns="measure", values="change %").round(1))
    display(pd.DataFrame(rows).round(3).query("significant"))

## 4. Emergency vehicles

In [ ]:
if res is not None and "ambulance_mean_travel_s" in res:
    display(res[res.scenario.isin(["emergency", "combined"])].pivot_table(index="scenario", columns="controller", values="ambulance_mean_travel_s")[order].round(1))

## 5. Benchmarks
* **Ablation:** one network agent switched off at a time (`python scripts/benchmark.py ablation`)
* **Demand sweep:** 3,000 to 5,500 vehicles/h, including the Webster engineering baseline (`sweep`)

In [ ]:
for csv, png, title in [("ablation_summary.csv", "ablation.png", "Ablation"), ("sweep_summary.csv", "sweep.png", "Demand sweep")]:
    t = nt.load_benchmark(csv)
    print(f"\n{title}:" if t is not None else f"\n{title}: not run yet")
    if t is not None:
        display(t)
        p = config.path("results", "benchmark", png)
        if os.path.exists(p): display(Image(p))